# 28.1 — Triplet Encoder + ANCE Hard Negative Mining (WJ 512)

Same encoder as nb28 but replaces **in-batch hard negatives** with **ANCE-style global hard negative mining**:
every `mine_every` epochs, encode the full corpus, build a temporary ANN index, and pick the
hardest non-GT corpus neighbor per query as an explicit negative. No B×B cross-similarity matrix —
each step is a clean (query, positive, hard_negative) triplet.

In [ ]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup, eval_recall, l1_simplex, load_dataset, load_dataset_normalized,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name   = "full"
out_dim        = 512
device         = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
THREADS        = 40
seed           = 42
batch_size     = 2048
epochs         = 100
lr             = 1e-3
weight_decay   = 1e-4
max_pos        = 30
temperature    = 0.07        # InfoNCE temperature
warmup_epochs  = 10         # in-batch InfoNCE warmup before ANCE mining starts
mine_every     = 2          # re-mine every N epochs (ANCE phase)
mine_k         = 2000       # ANN k for hard neg mining
n_hard_negs    = 5          # top-N non-GT ANCE negatives per query
eval_every     = 20         # mid-training R@50 diagnostic
candidate_ks   = [500, 1000] if dataset_name == "10k" else [1000, 2000]

es_patience  = 20
es_min_delta = 1e-4

METHOD_NAME   = "triplet_ance_wj_512"
NOTEBOOK_NAME = "28_1_triplet_ance_wj_512.ipynb"
OUT_PATH      = "/tmp/results_sota_triplet_ance_wj_512.pkl"
CKPT_PATH     = "/tmp/best_sota_triplet_ance_wj_512_full.pt"

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"dataset={dataset_name} | batch={batch_size} | epochs={epochs} | warmup={warmup_epochs} | "
      f"temp={temperature} | mine_every={mine_every} | mine_k={mine_k} | n_hard_negs={n_hard_negs}")

In [4]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums, qt_norm = load_dataset_normalized(dataset_name)

dataset=full | qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)
qt_norm loaded from cache (233773, 18220) in 10.5s


In [ ]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_sim_matrix(a, b, chunk=256):
    """Chunked (A,B) WJ matrix — avoids materializing full (A,B,D) tensor at once."""
    rows = []
    for i in range(0, len(a), chunk):
        ai   = a[i:i+chunk]                                            # (c, D)
        mins = torch.minimum(ai.unsqueeze(1), b.unsqueeze(0)).sum(-1)  # (c, B)
        maxs = torch.maximum(ai.unsqueeze(1), b.unsqueeze(0)).sum(-1).clamp(1e-10)
        rows.append(mins / maxs)
    return torch.cat(rows, dim=0)                                       # (A, B)

def inbatch_infonce_loss(zq, zp, temperature=0.07):
    """InfoNCE with all in-batch pairs. Label i = zp[i] is the positive for zq[i].
    2047 negatives per query — 2000× more signal than triplet with 1 negative."""
    sim    = wj_sim_matrix(zq, zp) / temperature   # (B, B)
    labels = torch.arange(len(zq), device=zq.device)
    loss   = F.cross_entropy(sim, labels)
    with torch.no_grad():
        acc = (sim.argmax(1) == labels).float().mean().item()
    return loss, acc

def hybrid_infonce_loss(zq, zp, z_negs, temperature=0.07):
    """
    InfoNCE with in-batch + ANCE negatives in the same denominator.
    zq, zp: (B, D); z_negs: (B, N, D)
    Logits = [in-batch sim (B,B) | ance sim (B,N)]; label[i] = i (diagonal = positive).
    """
    B, N   = zq.shape[0], z_negs.shape[1]
    sim_ib = wj_sim_matrix(zq, zp) / temperature                              # (B, B)
    sim_an = torch.stack([wj_sim(zq, z_negs[:, i]) for i in range(N)], dim=1) / temperature  # (B, N)
    logits = torch.cat([sim_ib, sim_an], dim=1)                               # (B, B+N)
    labels = torch.arange(B, device=zq.device)
    loss   = F.cross_entropy(logits, labels)
    with torch.no_grad():
        acc = (logits[:, :B].argmax(1) == labels).float().mean().item()
    return loss, acc

class PairDataset(Dataset):
    """Warmup phase: (q_id, p_id) pairs — no explicit negatives needed."""
    def __init__(self, gt_lookup, query_start, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            if qid < query_start: continue
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"  warmup pairs={len(self.pairs):,} | steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return torch.tensor(qid, dtype=torch.long), torch.tensor(pid, dtype=torch.long)

class ANCEDataset(Dataset):
    """ANCE phase: (q_id, p_id, neg_ids[N]) — pre-mined top-N hard negatives per query."""
    def __init__(self, gt_lookup, query_start, hard_negs, max_pos=30, n_negs=5):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            if qid < query_start or qid not in hard_negs: continue
            neg_list = hard_negs[qid][:]
            while len(neg_list) < n_negs: neg_list.append(neg_list[-1])
            neg_list = neg_list[:n_negs]
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid, neg_list))
        random.shuffle(self.pairs)
        n_q = len(set(p[0] for p in self.pairs))
        print(f"  ANCE pairs={len(self.pairs):,} | queries_with_neg={n_q} | steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid, neg_ids = self.pairs[idx]
        return (torch.tensor(qid, dtype=torch.long),
                torch.tensor(pid, dtype=torch.long),
                torch.tensor(neg_ids, dtype=torch.long))

class TripletEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)

def embed_all(model, qt, batch_size=4096):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(model(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def mine_hard_negatives(model, qt_norm, gt, query_start, k=mine_k, n_negs=n_hard_negs, threads=THREADS):
    """Mine top-n_negs non-GT corpus neighbors per query using current embeddings."""
    t0   = time.time()
    embs = embed_all(model, qt_norm)
    corpus_embs, query_embs = embs[:query_start], embs[query_start:]
    print(f"  encoded {len(qt_norm):,} in {time.time()-t0:.1f}s | ANN k={k}...", end=" ", flush=True)
    t1   = time.time()
    nbrs, _ = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=k, threads=threads)
    hard_negs = {}
    for q_offset in range(len(query_embs)):
        q_id   = query_start + q_offset
        gt_set = set(gt.get(q_id, []))
        found  = []
        for cand in nbrs[q_offset]:
            if int(cand) >= 0 and int(cand) not in gt_set:
                found.append(int(cand))
                if len(found) >= n_negs: break
        if found:
            hard_negs[q_id] = found
    print(f"{len(hard_negs)}/{len(query_embs)} queries got negs in {time.time()-t1:.1f}s "
          f"| total {time.time()-t0:.1f}s", flush=True)
    return hard_negs

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        print(f"{key} QPS={rr_metrics['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
    release_rerank_corpus()

In [4]:
device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")
print("Pre-loading vectors to cuda:7...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")

model = TripletEncoder(qt_norm.shape[1], out_dim)
model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.to(device)
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
best = float('inf')
t0_train = time.time()

# Early stopping state
viol_strike_count = 0
no_improve_count  = 0

def make_loader(hard_negs):
    ds = ANCEDataset(gt, query_start, hard_negs, max_pos=max_pos)
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True)

print(f"\n[init] Hard negative mining (k={mine_k}):")
hard_negs = mine_hard_negatives(model, qt_norm, gt, query_start, k=mine_k)
loader    = make_loader(hard_negs)

epoch_bar = tqdm(range(1, epochs + 1), desc="epochs", unit="ep")
for epoch in epoch_bar:
    just_remined = epoch > 1 and (epoch - 1) % mine_every == 0
    if just_remined:
        print(f"\n[ep{epoch:02d}] Re-mining (k={mine_k}):", flush=True)
        hard_negs = mine_hard_negatives(model, qt_norm, gt, query_start, k=mine_k)
        loader    = make_loader(hard_negs)

    model.train()
    tot_loss = tot_viol = steps = 0
    step_bar = tqdm(loader, desc=f"ep{epoch:02d}", leave=False, unit="step")
    for q_ids, p_ids, n_ids in step_bar:
        q = vecs_gpu[q_ids.to(vecs_device)].to(device)
        p = vecs_gpu[p_ids.to(vecs_device)].to(device)
        n = vecs_gpu[n_ids.to(vecs_device)].to(device)
        B = q.shape[0]
        z          = model(torch.cat([q, p, n]))
        zq, zp, zn = z[:B], z[B:2*B], z[2*B:]
        loss, n_viol = ance_triplet_loss(zq, zp, zn, margin=margin)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_viol += n_viol; steps += 1
        step_bar.set_postfix(loss=f"{float(loss.detach()):.4f}", viol=n_viol)
    sch.step()

    avg      = tot_loss / max(steps, 1)
    avg_viol = tot_viol / max(steps, 1)

    # Checkpoint on improvement
    if avg < best - es_min_delta:
        best = avg
        no_improve_count = 0
        torch.save(model.module.state_dict(), CKPT_PATH)
    else:
        no_improve_count += 1

    # Viol collapse check — only meaningful right after fresh re-mine
    if just_remined:
        if avg_viol < es_viol_threshold:
            viol_strike_count += 1
            print(f"  [ES] viol={avg_viol:.1f} after re-mine — strike {viol_strike_count}/{es_viol_strikes}", flush=True)
        else:
            viol_strike_count = 0

    elapsed = (time.time() - t0_train) / 60
    eta     = elapsed / epoch * (epochs - epoch)
    epoch_bar.set_postfix(loss=f"{avg:.4f}", best=f"{best:.4f}", viol=f"{avg_viol:.1f}", eta=f"{eta:.0f}m")
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | viol={avg_viol:.1f} | "
              f"{elapsed:.1f}min | eta={eta:.1f}min", flush=True)

    # Early stop
    if viol_strike_count >= es_viol_strikes:
        print(f"\nEarly stop ep{epoch}: violations < {es_viol_threshold} for {es_viol_strikes} consecutive re-mines → model saturated", flush=True)
        break
    if no_improve_count >= es_loss_patience:
        print(f"\nEarly stop ep{epoch}: no loss improvement for {es_loss_patience} epochs (best={best:.4f})", flush=True)
        break

print(f"\nTraining done. best={best:.4f} | saved {CKPT_PATH}")

Pre-loading vectors to cuda:7...
Loaded: 15.87 GB on cuda:7
DataParallel on 8 GPUs

[init] Hard negative mining (k=2000):
  Mining... 

/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

found 46141/46754 hard negs in 124.4s
  triplets=1,267,089 | queries_with_neg=44053 | steps/epoch=618


epochs:   0%|          | 0/75 [00:00<?, ?ep/s]

ep01:   0%|          | 0/618 [00:01<?, ?step/s]

...


[ep75] Re-mining (k=2000):
  Mining... 


0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

found 44483/46754 hard negs in 138.4s
  triplets=1,217,349 | queries_with_neg=42395 | steps/epoch=594


ep75:   0%|          | 0/594 [00:00<?, ?step/s]

epoch 75/75 | loss=0.0196 | viol=299.8 | 126.0min | eta=0.0min

Training done. best=0.0178 | saved /tmp/best_sota_triplet_ance_wj_512_full.pt


In [ ]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

R@10   = 0.2919
R@50   = 0.3352
R@100  = 0.3550
R@500  = 0.4299
QPS=3510.5
saved triplet_ance_wj_512 -> /tmp/results_sota_triplet_ance_wj_512.pkl
Corpus pre-loaded to GPU: 12.69 GB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*****************************************************



triplet_ance_wj_512_rerank_1000 R@10 = 0.8639
triplet_ance_wj_512_rerank_1000 R@50 = 0.7980
triplet_ance_wj_512_rerank_1000 R@100 = 0.7439
triplet_ance_wj_512_rerank_1000 R@500 = 0.5461
triplet_ance_wj_512_rerank_1000 QPS=382.1
saved triplet_ance_wj_512_rerank_1000 -> /tmp/results_sota_triplet_ance_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
****************************************************




triplet_ance_wj_512_rerank_2000 R@10 = 0.8842
triplet_ance_wj_512_rerank_2000 R@50 = 0.8227
triplet_ance_wj_512_rerank_2000 R@100 = 0.7700
triplet_ance_wj_512_rerank_2000 R@500 = 0.5720
triplet_ance_wj_512_rerank_2000 QPS=215.1
saved triplet_ance_wj_512_rerank_2000 -> /tmp/results_sota_triplet_ance_wj_512.pkl


In [6]:

# ── GPU Exact L1 Search ───────────────────────────────────────────────────────
# Diagnoses whether HNSW efSearch is the bottleneck or if embeddings themselves
# are the problem. WJ on L1-simplex is monotone with L1 distance, so exact L1
# nearest-neighbor search gives the true recall ceiling of this embedding space.

expected_rows = len(qt_norm)  # 233773 for full, 10000 for 10k
if 'embs' not in vars() or embs.shape[0] != expected_rows:
    print(f"Re-encoding (embs missing or wrong shape — expected {expected_rows} rows)...")
    enc = TripletEncoder(qt_norm.shape[1], out_dim)
    enc.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
    enc = nn.DataParallel(enc, device_ids=list(range(torch.cuda.device_count()))).to(device)
    embs = embed_all(enc, qt_norm)
    print(f"embs: {embs.shape}")
else:
    print(f"Using existing embs {embs.shape}")

corpus_embs_ex = embs[:query_start]
query_embs_ex  = embs[query_start:]
N, D = corpus_embs_ex.shape
Q    = len(query_embs_ex)
k    = 50

print(f"Exact search: {Q} queries × {N} corpus | D={D} | k={k}")
search_dev = torch.device("cuda:0")
corpus_gpu_ex = torch.from_numpy(corpus_embs_ex).to(search_dev)
print(f"Corpus on GPU: {corpus_gpu_ex.nbytes/1024**3:.2f} GB")

chunk_size = 64   # 64×187K×512×4 ≈ 23 GB intermediate — fine on A100
all_idx = []
t0 = time.time()
with torch.no_grad():
    for s in range(0, Q, chunk_size):
        q = torch.from_numpy(query_embs_ex[s:s+chunk_size]).to(search_dev)
        l1 = (q.unsqueeze(1) - corpus_gpu_ex.unsqueeze(0)).abs_().sum(dim=2)
        all_idx.append(l1.topk(k, dim=1, largest=False).indices.cpu().numpy())
elapsed = time.time() - t0
qps_exact = Q / elapsed
nbrs_exact = np.vstack(all_idx)

metrics_ex = eval_recall(gt, nbrs_exact, query_start, k)
print(f"\n--- GPU Exact L1 ---")
for kk in [10, 50]:
    print(f"R@{kk:<4} = {metrics_ex[kk]:.4f}")
print(f"QPS  = {qps_exact:.0f}  ({elapsed:.1f}s total)")
print(f"\nHNSW R@10=0.2919  →  Exact R@10={metrics_ex[10]:.4f}")
print("If Exact ≈ HNSW: problem is embedding quality, not search approximation.")
print("If Exact >> HNSW: efSearch too small, increase it.")
del corpus_gpu_ex


Re-encoding (embs missing or wrong shape — expected 233773 rows)...


/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)


embs: (233773, 512)
Exact search: 46754 queries × 187019 corpus | D=512 | k=50
Corpus on GPU: 0.36 GB

--- GPU Exact L1 ---
R@10   = 0.2919
R@50   = 0.3353
QPS  = 848  (55.1s total)

HNSW R@10=0.2919  →  Exact R@10=0.2919
If Exact ≈ HNSW: problem is embedding quality, not search approximation.
If Exact >> HNSW: efSearch too small, increase it.
